# 03. VC regularization과 Pareto 진단

목표: collapsed representation의 variance/covariance를 측정하고, flow와 segmentation metric을 동시에 비교하는 Pareto 관점을 익힌다.

In [ ]:
import math
import statistics

def vc_report(embeddings, gamma=1.0):
    columns = list(zip(*embeddings))
    means = [statistics.mean(column) for column in columns]
    variances = [statistics.pvariance(column) for column in columns]
    stds = [math.sqrt(variance + 1e-8) for variance in variances]
    variance_penalty = sum(max(0.0, gamma - std) for std in stds)

    covariance_penalty = 0.0
    for left in range(len(columns)):
        for right in range(len(columns)):
            if left == right:
                continue
            covariance = sum(
                (row[left] - means[left]) * (row[right] - means[right])
                for row in embeddings
            ) / len(embeddings)
            covariance_penalty += covariance * covariance
    return {
        "mean_variance": sum(variances) / len(variances),
        "variance_penalty": variance_penalty,
        "covariance_penalty": covariance_penalty,
    }

In [ ]:
collapsed = [[1.0, 1.0, 1.0] for _ in range(5)]
diverse = [
    [-1.4, 0.0, 1.2],
    [-0.7, 1.3, 0.1],
    [0.0, -1.2, -1.1],
    [0.8, 0.7, 0.5],
    [1.4, -0.8, -0.4],
]
print("collapsed:", vc_report(collapsed))
print("diverse:  ", vc_report(diverse))

In [ ]:
strategies = [
    {"name": "flow head training", "epe": 13.52, "miou": 60.1},
    {"name": "flow fine-tuning", "epe": 2.71, "miou": 61.3},
    {"name": "epoch alternation", "epe": 4.54, "miou": 63.5},
    {"name": "batch alternation", "epe": 2.78, "miou": 67.1},
    {"name": "combined loss", "epe": 2.67, "miou": 67.1},
]

def dominates(left, right):
    # EPE는 낮을수록, mIoU는 높을수록 좋다.
    no_worse = left["epe"] <= right["epe"] and left["miou"] >= right["miou"]
    strictly_better = left["epe"] < right["epe"] or left["miou"] > right["miou"]
    return no_worse and strictly_better

pareto = [
    candidate
    for candidate in strategies
    if not any(
        dominates(other, candidate)
        for other in strategies
        if other is not candidate
    )
]
print("Pareto 전략:", [item["name"] for item in pareto])

## 운영 checklist

- [ ] content와 flow branch의 loss 및 gradient norm을 따로 기록한다.
- [ ] NaN 발생 전에 estimator norm, flow clipping 비율과 feature variance를 경보로 둔다.
- [ ] dataset별 sampling/repetition과 실제 본 sample 수를 기록한다.
- [ ] EPE 하나로 checkpoint를 고르지 않고 segmentation metric도 함께 본다.
- [ ] VC coefficient, warmup, flow start epoch와 task weight를 ablation한다.
- [ ] 여러 seed의 평균·표준편차와 실패 run을 보고한다.
- [ ] 같은 compute budget에서 single-task baseline과 비교한다.